# 노선×정류소(BusRoutespecificStopInformation) → DuckDB 적재

공공데이터포털 `BusRoutespecificStopInformation/getBusRoutespecificStopInformation`.

**가장 복잡한 API** — 시군구 단위 순회 필요 (전국 ~229개 시군구).

- `opr_ymd` = 오늘 - 1개월
- `ctpv_cd` + `sgg_cd` 조합 모두 순회 → **수십만 건** 예상, 소요시간 **수십 분~1시간**
- 테이블: `route_station` (PK = rte_id + sttn_seq)
- **emd_cd가 10자리 법정동 코드** → `region.region_cd`와 직접 JOIN 가능
- 시군구별 try/except로 일부 실패해도 진행 유지

In [1]:
# ============================================================
# 셀 1 - 환경 설정 + DB 헬퍼 정의
# ============================================================
import os
import time
import requests
import duckdb
import pandas as pd
from pathlib import Path
from contextlib import contextmanager
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv()
SERVICE_KEY = unquote(os.environ["MY_SERVICE_KEY"])

DB_PATH = "data/seoul.duckdb"
Path("data").mkdir(exist_ok=True)


@contextmanager
def db_open(read_only: bool = False):
    """짧게 연결하고 무조건 닫는 컨텍스트 매니저 (예외 시에도 안전)."""
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()


print(f"SERVICE_KEY 로드됨 (길이 {len(SERVICE_KEY)})")
print(f"DB: {DB_PATH} (헬퍼 준비됨)")

SERVICE_KEY 로드됨 (길이 88)
DB: data/seoul.duckdb (헬퍼 준비됨)


In [2]:
# ============================================================
# 셀 2 - Pydantic 모델 (공공데이터 표준 응답 + RouteStationItem)
# ============================================================
from typing import Generic, TypeVar, Optional
from pydantic import BaseModel, field_validator

T = TypeVar("T", bound=BaseModel)


class Header(BaseModel):
    resultCode: str
    resultMsg: str


class Items(BaseModel, Generic[T]):
    item: list[T] = []

    @field_validator("item", mode="before")
    @classmethod
    def _coerce_to_list(cls, v):
        if v is None or v == "":
            return []
        if isinstance(v, dict):
            return [v]
        return v


class Body(BaseModel, Generic[T]):
    items: Items[T] = Items[T]()
    dataType: str | None = None
    pageNo: int
    numOfRows: int
    totalCount: int


class Response_(BaseModel, Generic[T]):
    header: Header
    body: Body[T]


class PublicDataResponse(BaseModel, Generic[T]):
    Response: Response_[T]


class RouteStationItem(BaseModel):
    """노선×정류소 매핑 한 건.

    PK: (rte_id, sttn_seq) — 노선 내 순번으로 유일.
      - 순환 노선도 sttn_seq가 다르면 같은 sttn_id가 반복 가능
      - rte_id 8자리 = sgg_cd(5) + 노선번호(3) 형태 (예: '11110001' = 종로구 100번)
      - emd_cd 10자리 = region 테이블의 region_cd와 JOIN 가능
    """
    # 필수 (PK + 운행일자)
    rte_id: str
    sttn_seq: int
    opr_ymd: str

    # API 품질 이슈 대비 전부 Optional로 방어
    sttn_id: Optional[str] = None
    sttn_nm: Optional[str] = None
    rte_no: Optional[str] = None
    rte_nm: Optional[str] = None
    ctpv_cd: Optional[str] = None
    ctpv_nm: Optional[str] = None
    sgg_cd: Optional[str] = None
    sgg_nm: Optional[str] = None
    emd_cd: Optional[str] = None
    emd_nm: Optional[str] = None
    trfc_mns_se_cd: Optional[str] = None   # B=Bus


print("모델 로드 완료")

모델 로드 완료


In [3]:
# ============================================================
# 셀 3 - fetch 함수 (NO_DATA_FOUND 에러 처리 포함)
# ============================================================
ROUTE_STATION_URL = "https://apis.data.go.kr/1613000/BusRoutespecificStopInformation/getBusRoutespecificStopInformation"


def fetch_all_pages(
    url: str,
    item_model: type[T],
    extra_params: dict | None = None,
    num_rows: int = 1000,
    verbose: bool = False,
) -> list[T]:
    """공공데이터 표준 응답 API 전체 페이지 수집. NO_DATA_FOUND는 빈 결과로 취급.
    
    verbose=False 면 페이지별 로그 생략 (시군구 순회 시 로그 폭주 방지).
    """
    params_base = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_rows,
        "dataType": "JSON",
        **(extra_params or {}),
    }
    all_items: list[T] = []
    page = 1
    while True:
        res = requests.get(url, params={**params_base, "pageNo": page}, timeout=30)
        res.raise_for_status()
        payload = res.json()

        # 에러 응답 선처리 (데이터 없는 시군구)
        if "Error" in payload:
            err = payload["Error"]
            code = err.get("code")
            msg = err.get("message", "")
            if msg == "NO_DATA_FOUND" or code == "50":
                return all_items
            raise RuntimeError(f"API error: {code} {msg}")

        parsed = PublicDataResponse[item_model].model_validate(payload)
        header = parsed.Response.header
        if header.resultCode not in ("00", "200"):
            raise RuntimeError(f"API error: {header.resultCode} {header.resultMsg}")

        body = parsed.Response.body
        items = body.items.item
        all_items.extend(items)

        if verbose:
            print(f"    page {page}: +{len(items)} (누적 {len(all_items)}/{body.totalCount})")

        if len(all_items) >= body.totalCount or not items:
            break
        page += 1
        time.sleep(0.1)   # 페이지 간 간격

    return all_items


print("fetch 함수 준비 완료")

fetch 함수 준비 완료


In [ ]:
# ============================================================
# 셀 4 - 파라미터 준비: opr_ymd + 시군구 목록 + 테이블 생성 + 재개 목록
# ============================================================
# 재개(resume) 지원:
#   - 테이블을 먼저 CREATE IF NOT EXISTS
#   - 이미 적재된 (ctpv_cd, sgg_cd) 조합을 조회해서 skip 목록으로 사용
#   - 중간에 중단해도 다시 실행하면 남은 시군구만 이어서 처리
#
# ⚠️ 정규화: ctpv_nm / sgg_nm / emd_nm은 region에서 JOIN으로 가져오므로 저장 안 함.

opr_ymd = (date.today() - relativedelta(months=1)).strftime("%Y%m%d")
print(f"운행일자 opr_ymd = {opr_ymd}")

with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS route_station (
        rte_id          TEXT,        -- 노선 ID (8자리)
        sttn_seq        INTEGER,     -- 노선 내 순번
        sttn_id         TEXT,        -- 정류소 ID
        sttn_nm         TEXT,        -- 정류소명 (정류소 마스터 없어 유지)
        rte_no          TEXT,        -- 노선 번호
        rte_nm          TEXT,        -- 노선명
        ctpv_cd         TEXT,        -- 시도 코드 (2자리) → region JOIN으로 이름
        sgg_cd          TEXT,        -- 시군구 코드 (5자리) → region JOIN으로 이름
        emd_cd          TEXT,        -- 법정동 코드 (10자리) → region.region_cd JOIN
        trfc_mns_se_cd  TEXT,        -- 교통수단구분 (B=Bus)
        opr_ymd         TEXT,
        PRIMARY KEY (rte_id, sttn_seq)
    )
    """)

with db_open(read_only=True) as con:
    sgg_list_df = con.execute("""
        SELECT sido_cd,
               sgg_cd,
               sido_cd || sgg_cd AS ctpv_sgg_5,
               locatadd_nm       AS sgg_nm
        FROM region
        WHERE sgg_cd <> '000'
          AND umd_cd = '000'
          AND ri_cd = '00'
        ORDER BY sido_cd, sgg_cd
    """).df()

    done_df = con.execute("""
        SELECT DISTINCT ctpv_cd, sgg_cd FROM route_station
    """).df()

done_set = set(zip(done_df["ctpv_cd"], done_df["sgg_cd"]))
total = len(sgg_list_df)
remaining = sgg_list_df[
    ~sgg_list_df.apply(lambda r: (r["sido_cd"], r["ctpv_sgg_5"]) in done_set, axis=1)
]

print(f"\n전국 시군구: {total}개")
print(f"이미 완료: {total - len(remaining)}개 (재개 시 skip)")
print(f"이번에 처리할 남은 시군구: {len(remaining)}개")

if len(remaining) > 0:
    print(f"\n처리 예정 앞 5개:")
    print(remaining.head())

In [ ]:
# ============================================================
# 셀 5 - 전국 시군구 순회 → 시군구별로 즉시 DB 저장
# ============================================================
# 각 시군구 fetch 직후 바로 DB INSERT → 중간 중단 시에도 진행분 보존.
# PK 기준 ON CONFLICT DO NOTHING → 재실행해도 중복 INSERT 안 됨.
# INSERT에 이름 컬럼(ctpv_nm/sgg_nm/emd_nm) 제외 — 정규화 스키마.

failed: list[tuple[str, str, str]] = []
started_at = datetime.now()
processed_total = 0

for i, row in enumerate(remaining.itertuples(index=False), start=1):
    ctpv_cd = row.sido_cd
    sgg_5   = row.ctpv_sgg_5
    sgg_nm  = row.sgg_nm

    try:
        items = fetch_all_pages(
            ROUTE_STATION_URL,
            RouteStationItem,
            extra_params={
                "opr_ymd": opr_ymd,
                "ctpv_cd": ctpv_cd,
                "sgg_cd":  sgg_5,
            },
        )

        if items:
            df_sgg = pd.DataFrame([it.model_dump() for it in items])
            df_sgg = df_sgg.dropna(subset=["rte_id", "sttn_seq"])
            df_sgg = df_sgg.drop_duplicates(subset=["rte_id", "sttn_seq"])
        else:
            df_sgg = pd.DataFrame()

        if len(df_sgg) > 0:
            with db_open() as con:
                con.register("df_view", df_sgg)
                con.execute("""
                    INSERT INTO route_station
                    SELECT rte_id, sttn_seq, sttn_id, sttn_nm,
                           rte_no, rte_nm,
                           ctpv_cd, sgg_cd, emd_cd,
                           trfc_mns_se_cd, opr_ymd
                    FROM df_view
                    ON CONFLICT (rte_id, sttn_seq) DO NOTHING
                """)
                con.unregister("df_view")

        processed_total += len(df_sgg)
        status = f"+{len(df_sgg)}건"
    except Exception as e:
        failed.append((ctpv_cd, sgg_5, str(e)[:100]))
        status = f"FAILED ({type(e).__name__})"

    if i % 10 == 0 or i == len(remaining):
        elapsed = datetime.now() - started_at
        pct = i / len(remaining) * 100
        eta_sec = elapsed.total_seconds() / i * (len(remaining) - i)
        print(f"[{i:3d}/{len(remaining)}] {pct:5.1f}% | {sgg_nm:25s} {status:15s} "
              f"| 이번 실행 적재 {processed_total:>7,}건 | 경과 {elapsed} | 잔여 ~{int(eta_sec//60)}분")

    time.sleep(0.15)

print(f"\n✅ 수집 완료")
print(f"   이번 실행에서 {processed_total:,}건 적재")
print(f"   실패 시군구: {len(failed)}개")
print(f"   총 소요시간: {datetime.now() - started_at}")
if failed:
    print(f"\n실패 목록:")
    for f in failed[:10]:
        print(f"   {f}")

with db_open(read_only=True) as con:
    grand_total = con.execute("SELECT COUNT(*) FROM route_station").fetchone()[0]
print(f"\n📊 route_station 총 누적: {grand_total:,}건")

In [6]:
# ============================================================
# 셀 6 - (선택) 테이블 초기화
# ============================================================
# 처음부터 다시 수집하고 싶을 때만 수동 실행.
# 시군구별 즉시 저장 패턴이라 평소에는 실행할 필요 없음.

RESET = False   # True로 바꾸고 실행하면 테이블 비움

if RESET:
    with db_open() as con:
        con.execute("DELETE FROM route_station")
        cnt = con.execute("SELECT COUNT(*) FROM route_station").fetchone()[0]
    print(f"⚠️  테이블 초기화됨: {cnt}건 (0이어야 함)")
else:
    print("RESET=False → 아무 작업 안 함. 초기화하려면 위 변수를 True로 변경 후 실행")

RESET=False → 아무 작업 안 함. 초기화하려면 위 변수를 True로 변경 후 실행


In [ ]:
# ============================================================
# 셀 7 - 실패 시군구 재시도 (필요 시)
# ============================================================
if not failed:
    print("실패 시군구 없음 — 재시도 불필요")
else:
    print(f"재시도할 실패 시군구: {len(failed)}개")
    retry_failed: list[tuple[str, str, str]] = []

    for ctpv_cd, sgg_5, prev_err in failed:
        try:
            items = fetch_all_pages(
                ROUTE_STATION_URL,
                RouteStationItem,
                extra_params={
                    "opr_ymd": opr_ymd,
                    "ctpv_cd": ctpv_cd,
                    "sgg_cd":  sgg_5,
                },
            )
            if items:
                df_sgg = pd.DataFrame([it.model_dump() for it in items])
                df_sgg = df_sgg.dropna(subset=["rte_id", "sttn_seq"])
                df_sgg = df_sgg.drop_duplicates(subset=["rte_id", "sttn_seq"])

                with db_open() as con:
                    con.register("df_view", df_sgg)
                    con.execute("""
                        INSERT INTO route_station
                        SELECT rte_id, sttn_seq, sttn_id, sttn_nm,
                               rte_no, rte_nm,
                               ctpv_cd, sgg_cd, emd_cd,
                               trfc_mns_se_cd, opr_ymd
                        FROM df_view
                        ON CONFLICT (rte_id, sttn_seq) DO NOTHING
                    """)
                    con.unregister("df_view")
                print(f"  ✅ {ctpv_cd}/{sgg_5}: +{len(df_sgg)}건")
            else:
                print(f"  ⚠️  {ctpv_cd}/{sgg_5}: 빈 응답")
        except Exception as e:
            retry_failed.append((ctpv_cd, sgg_5, str(e)[:100]))
            print(f"  ❌ {ctpv_cd}/{sgg_5}: 다시 실패 — {e}")
        time.sleep(0.2)

    failed = retry_failed
    print(f"\n재시도 후 여전히 실패: {len(failed)}개")

In [ ]:
# ============================================================
# 셀 8 - 검증 (정규화 테이블: 이름은 region JOIN으로 복원)
# ============================================================
with db_open(read_only=True) as con:
    print("=== 시도별 노선×정류소 매핑 수 ===")
    print(con.execute("""
        SELECT rs.ctpv_cd,
               sido.locallow_nm           AS ctpv_nm,
               COUNT(*)                   AS mapping_cnt,
               COUNT(DISTINCT rs.sttn_id) AS unique_stops,
               COUNT(DISTINCT rs.rte_id)  AS unique_routes
        FROM route_station rs
        LEFT JOIN region sido
          ON sido.sido_cd = rs.ctpv_cd
         AND sido.sgg_cd  = '000'
         AND sido.umd_cd  = '000'
        GROUP BY rs.ctpv_cd, sido.locallow_nm
        ORDER BY mapping_cnt DESC
    """).df())

    print("\n=== 동작구 정류소 수 ===")
    print(con.execute("""
        SELECT COUNT(*) AS mapping_cnt,
               COUNT(DISTINCT sttn_id) AS unique_stops
        FROM route_station
        WHERE sgg_cd = '11590'
    """).df())

    print("\n=== JOIN 테스트: 동작구 상도동 일대 노선×정류소 ===")
    print(con.execute("""
        SELECT rs.rte_no, rs.rte_nm, rs.sttn_nm,
               r.locatadd_nm AS emd_full_nm
        FROM route_station rs
        JOIN region r ON rs.emd_cd = r.region_cd
        WHERE r.sido_cd = '11'
          AND r.sgg_cd  = '590'
          AND r.locallow_nm LIKE '상도%'
        LIMIT 10
    """).df())